## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [2]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
#!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
#!CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python --force-reinstall --no-cache-dir -q

!CMAKE_ARGS="-DLLAMA_METAL=on" pip install --force-reinstall --no-cache-dir llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 51.7 MB/s  0:00:00 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 53.6 MB/s  0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.16-cp313-cp313-macosx_26_0_arm64.whl size=3842966 sha256=cea0d92fa77a9163e2fb8daa00a70b86687f38e6f7e16ece43706460cd8c663c
  Stored in directory: /private/var/folders/qx/cd3rb17s6q93jfnkvf109f6r0000gn/T/pip-ephem-wheel-cache-u7etopek/wheels/af/3a/b6/445d9f4ccadd3ed923d55af8f055f2ccd217c66f09c834f0d8
Successfully built llama-cpp-python
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Successfully uninstalled typing_extensions-4.15.032m0/6 [typing-extensions]
  Attempting uninstall: numpy━━━━━━━━━━━━━━━ 0/6 [typing-extensions]
   

In [3]:
# For installing the libraries & downloading models from HF Hub
%pip install huggingface_hub pandas tiktoken pymupdf langchain langchain-community chromadb sentence-transformers numpy -q

Note: you may need to restart the kernel to use updated packages.


In [4]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

#### Downloading and Loading the model

In [5]:
#mistral-7b-instruct-v0.2.Q5_K_S.gguf suggested by https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF
model_path = hf_hub_download(repo_id="TheBloke/Mistral-7B-Instruct-v0.2-GGUF", filename="mistral-7b-instruct-v0.2.Q5_K_S.gguf")

In [101]:
llm = Llama(
    model_path=model_path, 
    n_ctx=4096, 
    n_threads=8, 
    n_gpu_layers=35, 
    temperature=0.7,
    repeat_penalty=1.1)

llama_model_load_from_file_impl: using device Metal (Apple M1 Max) - 19409 MiB free
llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /Users/parvatam/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF/snapshots/3a6fbf4a41a1d52e415a4958cde6856d34b2db93/mistral-7b-instruct-v0.2.Q5_K_S.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader:

#### Response

In [12]:
from IPython.display import Markdown, display

def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return Markdown(model_output['choices'][0]['text'])

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [13]:
sepsis_query1="What is the protocol for managing sepsis in a critical care unit?"
response(sepsis_query1)
# checking llm response

Llama.generate: 2 prefix-match hit, remaining 14 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =     375.91 ms /    14 tokens (   26.85 ms per token,    37.24 tokens per second)
llama_perf_context_print:        eval time =    2821.39 ms /   127 runs   (   22.22 ms per token,    45.01 tokens per second)
llama_perf_context_print:       total time =    3255.41 ms /   141 tokens
llama_perf_context_print:    graphs reused =        122




Sepsis is a life-threatening condition that can arise from an infection, and it is important to recognize and manage it promptly in a critical care unit. The following steps outline the general protocol for managing sepsis in a critical care unit:

1. Early recognition: Recognize the signs and symptoms of sepsis, which may include fever, chills, rapid heart rate, rapid breathing, confusion, and low blood pressure. Suspect sepsis in any patient with suspected or confirmed infection who is showing signs of organ dysfunction.
2. Resuscitation: Begin resusc

##### above code output shows llm is responding with some data

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [14]:

appendicitis_query2="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

response(appendicitis_query2)

# checking llm response for query2

Llama.generate: 2 prefix-match hit, remaining 32 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =     255.00 ms /    32 tokens (    7.97 ms per token,   125.49 tokens per second)
llama_perf_context_print:        eval time =    2770.81 ms /   127 runs   (   21.82 ms per token,    45.83 tokens per second)
llama_perf_context_print:       total time =    3069.74 ms /   159 tokens
llama_perf_context_print:    graphs reused =        122




Appendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right side of the abdomen. The symptoms of appendicitis can vary from person to person, but the following are the most common:

1. Abdominal pain: The pain is usually sharp, constant, and localized in the lower right abdomen. It may start as a mild discomfort, but it can quickly worsen and become severe.
2. Loss of appetite: People with appendicitis may lose their appetite and feel nauseous.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [15]:
hairLoss_query3="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

response(hairLoss_query3)

Llama.generate: 4 prefix-match hit, remaining 34 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =     384.00 ms /    34 tokens (   11.29 ms per token,    88.54 tokens per second)
llama_perf_context_print:        eval time =    2804.02 ms /   127 runs   (   22.08 ms per token,    45.29 tokens per second)
llama_perf_context_print:       total time =    3245.50 ms /   161 tokens
llama_perf_context_print:    graphs reused =        122




Sudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles, leading to hair loss in small, round patches on the scalp, beard, or other areas of the body. The exact cause of alopecia areata is not known, but it is believed to be related to a combination of genetic and environmental factors.

There are several treatments and solutions for addressing sudden patchy hair loss:

1. Topical corticosteroids: These are anti-inflammatory medications that can be applied

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [16]:
brainTissue_query4="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

response(brainTissue_query4)

Llama.generate: 2 prefix-match hit, remaining 28 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =     283.69 ms /    28 tokens (   10.13 ms per token,    98.70 tokens per second)
llama_perf_context_print:        eval time =    2821.15 ms /   127 runs   (   22.21 ms per token,    45.02 tokens per second)
llama_perf_context_print:       total time =    3173.33 ms /   155 tokens
llama_perf_context_print:    graphs reused =        122




There is no one-size-fits-all answer to this question, as the specific treatment recommendations for a person with a brain injury depend on the severity and location of the injury, as well as the individual's overall health and medical history. However, I can provide some general information about common treatments and interventions that may be used to help manage the symptoms and promote recovery after a brain injury.

1. Medical management: Depending on the severity of the injury, the person may require hospitalization and intensive care to manage life-threatening conditions such as brain swelling, bleeding,

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [17]:
legFracture_query5="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

response(legFracture_query5)

Llama.generate: 2 prefix-match hit, remaining 35 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =     816.26 ms /    35 tokens (   23.32 ms per token,    42.88 tokens per second)
llama_perf_context_print:        eval time =    2868.41 ms /   127 runs   (   22.59 ms per token,    44.28 tokens per second)
llama_perf_context_print:       total time =    3754.78 ms /   162 tokens
llama_perf_context_print:    graphs reused =        122




First and foremost, if you suspect that someone has fractured their leg during a hiking trip, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:

1. Keep the person calm and still: Encourage the person to remain calm and avoid moving the injured leg as much as possible to prevent further damage or discomfort.
2. Assess the injury: Check the leg for signs of deformity, swelling, or bruising. If the person is unable to put weight on the leg or is experiencing severe pain, it's likely that

## Question Answering using LLM with Prompt Engineering

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [18]:
sepsis_prompt="Answer this question as a medical researcher to help other researchers."
response(sepsis_prompt+sepsis_query1)

Llama.generate: 1 prefix-match hit, remaining 28 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =     423.87 ms /    28 tokens (   15.14 ms per token,    66.06 tokens per second)
llama_perf_context_print:        eval time =    2855.90 ms /   127 runs   (   22.49 ms per token,    44.47 tokens per second)
llama_perf_context_print:       total time =    3338.06 ms /   155 tokens
llama_perf_context_print:    graphs reused =        122




As a medical researcher, I would recommend the following protocol for managing sepsis in a critical care unit, based on current evidence-based guidelines and best practices:

1. Early recognition and diagnosis: Sepsis should be suspected in any patient with suspected or confirmed infection and signs of organ dysfunction. Use the Sequential Organ Failure Assessment (SOFA) score or Quick Sequential Organ Failure Assessment (qSOFA) score to identify patients at risk for sepsis.
2. Fluid resuscitation: Aggressively resuscitate patients with intravenous fluids

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [21]:
appendicitis_prompt="Answer this question as a expert medical surgeon to another surgeon who already know about appendicitis."

response(appendicitis_prompt+appendicitis_query2)

Llama.generate: 14 prefix-match hit, remaining 41 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =     531.06 ms /    41 tokens (   12.95 ms per token,    77.20 tokens per second)
llama_perf_context_print:        eval time =    2858.04 ms /   127 runs   (   22.50 ms per token,    44.44 tokens per second)
llama_perf_context_print:       total time =    3462.01 ms /   168 tokens
llama_perf_context_print:    graphs reused =        122




As a medical surgeon, I would explain that the common symptoms for appendicitis include:

1. Periumbilical or right lower quadrant abdominal pain that begins as a vague discomfort and progresses to sharp, localized pain.
2. Anorexia and nausea, which may lead to vomiting.
3. Low-grade fever, often below 101°F (38.3°C).
4. Loss of appetite.
5. Rebound tenderness, which is pain when the abdomen is pressed gently and then released.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [22]:
hairLoss_prompt="Answer this question as a dermatologist to another dermatologist concisely."
response(hairLoss_prompt+hairLoss_query3)

Llama.generate: 6 prefix-match hit, remaining 50 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =    1252.68 ms /    50 tokens (   25.05 ms per token,    39.91 tokens per second)
llama_perf_context_print:        eval time =    2862.90 ms /   127 runs   (   22.54 ms per token,    44.36 tokens per second)
llama_perf_context_print:       total time =    4197.74 ms /   177 tokens
llama_perf_context_print:    graphs reused =        122




As a dermatologist, I would suggest the following potential causes and treatments for sudden, patchy hair loss:

1. Alopecia Areata: An autoimmune disorder that causes hair loss in circular patches. Treatment options include topical corticosteroids, intralesional triamcinolone, systemic corticosteroids, immunomodulators, and phototherapy.
2. Tinea Capitis: A fungal infection of the scalp that can cause patchy hair loss. Treatment involves antifungal shampoos and oral antifung

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [24]:
brainTissue_prompt="Answer this question as a expert neurologist to a PHD student concisely."

response(brainTissue_prompt+brainTissue_query4)

Llama.generate: 15 prefix-match hit, remaining 32 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =     273.37 ms /    32 tokens (    8.54 ms per token,   117.06 tokens per second)
llama_perf_context_print:        eval time =    2783.76 ms /   127 runs   (   21.92 ms per token,    45.62 tokens per second)
llama_perf_context_print:       total time =    3111.31 ms /   159 tokens
llama_perf_context_print:    graphs reused =        122




As a neurology expert, I would recommend the following treatments for a person with a brain injury, depending on the severity and specific symptoms:

1. Acute care: For severe brain injuries, immediate care includes managing airway, breathing, and circulation, controlling intracranial pressure, and preventing or treating seizures.
2. Rehabilitation: Physical, occupational, and speech therapy can help improve motor function, strength, coordination, and communication skills.
3. Medications: Depending on the symptoms, medications may be prescribed to manage seizures, control pain, improve cognitive

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [26]:
legFracture_prompt="Answer this question as a sports specialist to a Athelete in less words"
response(legFracture_prompt+legFracture_query5)

Llama.generate: 13 prefix-match hit, remaining 39 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =     419.44 ms /    39 tokens (   10.75 ms per token,    92.98 tokens per second)
llama_perf_context_print:        eval time =    2784.02 ms /   127 runs   (   21.92 ms per token,    45.62 tokens per second)
llama_perf_context_print:       total time =    3263.55 ms /   166 tokens
llama_perf_context_print:    graphs reused =        122




1. Immediate care: R.I.C.E. - Rest, Ice, Compression, Elevation.
2. Seek medical attention: Assess the severity and potential complications.
3. Immobilize the leg: Use a splint or cast to prevent movement.
4. Pain management: Over-the-counter pain relievers or prescription medication.
5. Follow-up care: Regular check-ups with a healthcare professional.
6. Rehabilitation: Physical therapy to regain strength and mobility.
7. Nutrition and hydration: Proper

## Data Preparation for RAG

### Loading the Data

In [28]:
# Loading the PDF file
loader = PyMuPDFLoader("medical_diagnosis_manual.pdf")
pages = loader.load()

### Data Overview

#### Checking the first 5 pages

In [29]:
# Data Overview - Display first 5 pages
for page in pages[:5]:
    print(f"Page {page.metadata['page']}: {page.page_content[:200]}...")
    print("-" * 80)

Page 0: parvatamaditya@gmail.com
HGWQDEB19Y
nt for personal use by parvatamaditya@
shing the contents in part or full is liable...
--------------------------------------------------------------------------------
Page 1: parvatamaditya@gmail.com
HGWQDEB19Y
This file is meant for personal use by parvatamaditya@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action....
--------------------------------------------------------------------------------
Page 2: Table of Contents
1
Front    ..............................................................................................................................................................................
--------------------------------------------------------------------------------
Page 3: 491
Chapter 44. Foot & Ankle Disorders    .....................................................................................................................................
502
Chapter 45. Tumors o...
------

#### Checking the number of pages

In [30]:
# Show total number of pages
print(f"Total number of pages: {len(pages)}")


Total number of pages: 4114


### Data Chunking

In [33]:
# Add tokenizer encoding
encoding = tiktoken.get_encoding("cl100k_base")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,  # Number of tokens per chunk
    chunk_overlap=50,  # Token overlap between chunks
    length_function=lambda text: len(encoding.encode(text)),  
    add_start_index=True,
)
# Chunk the documents
pdf_chunks = loader.load_and_split(text_splitter)
print(f"Total chunks created: {len(pdf_chunks)}")

Total chunks created: 10992


In [ ]:
print(f"chunk-1: {pdf_chunks[0].page_content}")

print(f"chunk-2: {pdf_chunks[1].page_content}")

print(f"chunk-3: {pdf_chunks[2].page_content}")

chunk-1: parvatamaditya@gmail.com
HGWQDEB19Y
nt for personal use by parvatamaditya@
shing the contents in part or full is liable
chunk-2: parvatamaditya@gmail.com
HGWQDEB19Y
This file is meant for personal use by parvatamaditya@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
chunk-3: Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ...........................................................................................................................................................................................
53
1 - Nutrit

In [37]:
print(f"chunk-4: {pdf_chunks[3].page_content}")

chunk-4: Chapter 17. Malabsorption Syndromes    ..............................................................................................................................
225
Chapter 18. Irritable Bowel Syndrome    ................................................................................................................................
229
Chapter 19. Inflammatory Bowel Disease    .........................................................................................................................
241
Chapter 20. Diverticular Disease    ...........................................................................................................................................
246
Chapter 21. Anorectal Disorders    ............................................................................................................................................
254
Chapter 22. Tumors of the GI Tract    ..................................................................................

#### in the prints of above chuks content, we clearly see there are overlapping like expected.

### Embedding

In [40]:
# Create embeddings
# embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2") this gave warning of deprecation, so usign the latest recomended one

from langchain_community.embeddings import HuggingFaceEmbeddings

# Create embeddings
embedding_function = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

In [44]:
embedding_1 = embedding_function.embed_documents([pdf_chunks[0].page_content])
print(f"Embedding for chunk-1: {embedding_1[:5]}...")
embedding_2 = embedding_function.embed_documents([pdf_chunks[1].page_content])
print(f"Embedding for chunk-2: {embedding_2[:5]}...")

# checking embeding length
print(f"Length of embedding for chunk-1: {len(embedding_1[0])}")
print(f"Length of embedding for chunk-2: {len(embedding_2[0])}")

Embedding for chunk-1: [[-0.08239952474832535, 0.07843741774559021, 0.021244268864393234, -0.04620516672730446, 0.02556600421667099, -0.017492128536105156, 0.05670662596821785, 0.055810071527957916, -0.0157490037381649, 0.006131362169981003, 0.0956079438328743, -0.0473058857023716, 0.016143904998898506, -0.05628195032477379, -0.03319653868675232, -0.0749211311340332, -0.024146879091858864, -0.03490043804049492, -0.024814413860440254, 0.028518008068203926, -0.0316605381667614, -0.027707181870937347, -0.031129086390137672, 0.024569416418671608, -0.012330123223364353, 0.044099945574998856, -0.035157836973667145, 0.016495605930685997, 0.024283776059746742, -0.060978759080171585, 0.036747608333826065, -0.01482254546135664, 0.06476065516471863, -0.0102024981752038, 0.007888584397733212, 0.0732211172580719, -0.06642107665538788, -0.02166876755654812, -0.007506138179451227, -0.040105853229761124, -0.039160579442977905, -0.07042329013347626, -0.05361636355519295, 0.03300012648105621, 0.03599671

### Vector Database

In [46]:
# Create and persist the vector database
vectordb = Chroma.from_documents(
    documents=pdf_chunks,
    embedding=embedding_function,
    persist_directory="medical_db"
)
vectordb.persist()

/var/folders/qx/cd3rb17s6q93jfnkvf109f6r0000gn/T/ipykernel_26377/1223088613.py:7: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


In [48]:
vectordb.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={'normalize_embeddings': True}, multi_process=False, show_progress=False)

In [50]:
vectordb.similarity_search("diagnosis for fracture",k=3)

[Document(metadata={'creationdate': '2012-06-15T05:44:40+00:00', 'start_index': -1, 'trapped': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'modDate': 'D:20250913161404Z', 'subject': '', 'total_pages': 4114, 'keywords': '', 'page': 3391, 'moddate': '2025-09-13T16:14:04+00:00', 'creationDate': 'D:20120615054440Z', 'creator': 'Atop CHM to PDF Converter', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'author': '', 'source': 'medical_diagnosis_manual.pdf', 'format': 'PDF 1.7', 'file_path': 'medical_diagnosis_manual.pdf'}, page_content='and for fractures that occur during birth, see p. 2774.)\nFractures are cracks in bones. Symptoms include pain, swelling, ecchymosis, crepitation,\ndeformity, and abnormal motion. Occasional complications include fat embolism, arterial injury,\ncompartment syndrome, nerve injuries, and infection. Diagnosis is by clinical criteria and\nusually plain x-rays. Treatment involves analgesics, immobilization, and sometimes sur

### Retriever

In [47]:
# Create retriever
retriever = vectordb.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [54]:
#test retrievar
test_rettriever = retriever.get_relevant_documents("how to treat sepsis",k=3)

display(Markdown(f"### Retrieved {len(test_rettriever)} documents:"))
for i, doc in enumerate(test_rettriever):
    display(Markdown(f"**Document {i+1}:** {doc.page_content[:500]}..."))


### Retrieved 3 documents:

**Document 1:** Parenteral antibiotics should be given after specimens of blood, body fluids, and wound sites have been
taken for Gram stain and culture. Very prompt empiric therapy, started immediately after suspecting
sepsis, is essential and may be lifesaving. Antibiotic selection requires an educated guess based on the
suspected source, clinical setting, knowledge or suspicion of causative organisms and of sensitivity
patterns common to that specific inpatient unit, and previous culture results.
One regimen...

**Document 2:** can approximate bone marrow NSP levels. I:T ratios of > 0.80 correlate with NSP depletion and death;
such a ratio may identify neonates who might benefit from granulocyte transfusion.
Treatment
• Antibiotic therapy
• Supportive therapy
Because sepsis may manifest with non-specific clinical signs and its effects may be devastating, rapid
empiric antibiotic therapy is recommended (see p. 1182); drugs are later adjusted according to
sensitivities and the site of infection. If bacterial cultures sho...

**Document 3:** intracranial surgery within 2 mo, acute trauma with a risk of bleeding, and intracranial neoplasm. Risk-
benefit assessment is required in other patients with increased risk of serious bleeding (eg, with
thrombocytopenia or recent GI bleeding, receiving concurrent heparin, or with recent aspirin or other
anticoagulant use).
Other emerging therapies for severe sepsis include cooling for hyperthermia and early treatment of renal
failure (eg, with continuous venovenous hemofiltration).
Trials of mo...

### System and User Prompt Template

In [57]:
qna_system_message = """
You are an assistant whose work is to read the manual and provide the appropriate answers from the context.
User input will have the context required by you to answer user questions.
This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Please answer only using the context provided in the input. Do not mention anything about the context in your final answer.

If the answer is not found in the context, respond "I don't know".
"""

In [58]:
qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""

### Response Function

In [76]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [82]:
display(Markdown(generate_rag_response(sepsis_query1)))

Llama.generate: 145 prefix-match hit, remaining 1195 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =    3078.17 ms /  1195 tokens (    2.58 ms per token,   388.22 tokens per second)
llama_perf_context_print:        eval time =    3112.11 ms /   127 runs   (   24.50 ms per token,    40.81 tokens per second)
llama_perf_context_print:       total time =    6232.14 ms /  1322 tokens
llama_perf_context_print:    graphs reused =        122


Based on the context, the protocol for managing sepsis in a critical care unit includes:
1. Suspecting sepsis or septic shock based on symptoms such as shaking chills, persistent fever, altered sensorium, hypotension, and GI symptoms.
2. Obtaining cultures of blood and any other appropriate specimens.
3. Giving empiric antibiotics after appropriate cultures are obtained.
4. Adjusting antibiotics according to the results of culture and susceptibility testing.
5. Surgically draining any abscesses.
6

#####  Analysis till now
* the first phase gave generic anaswers from llm
* second phase, we gave some prompts to direct llm to think like the persona given
* in the above phase(3) we gave context from merck manual..which is giving answers from the context.. which is a RAG impelemtation

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [78]:
display(Markdown(generate_rag_response(appendicitis_query2)))

Llama.generate: 145 prefix-match hit, remaining 1380 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =    3453.86 ms /  1380 tokens (    2.50 ms per token,   399.55 tokens per second)
llama_perf_context_print:        eval time =    3247.59 ms /   127 runs   (   25.57 ms per token,    39.11 tokens per second)
llama_perf_context_print:       total time =    6757.48 ms /  1507 tokens
llama_perf_context_print:    graphs reused =        122


The common symptoms for appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point. Additional signs are pain felt in the right lower quadrant with palpation of the left lower quadrant (Rovsing sign), an increase in pain from passive extension of the right hip joint that stretches the iliopsoas muscle

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [79]:
display(Markdown(generate_rag_response(hairLoss_query3)))

Llama.generate: 145 prefix-match hit, remaining 1374 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =    3432.59 ms /  1374 tokens (    2.50 ms per token,   400.28 tokens per second)
llama_perf_context_print:        eval time =    3261.73 ms /   127 runs   (   25.68 ms per token,    38.94 tokens per second)
llama_perf_context_print:       total time =    6735.13 ms /  1501 tokens
llama_perf_context_print:    graphs reused =        122


Based on the context, the possible causes for sudden patchy hair loss could be alopecia areata or tinea capitis. The effective treatments for these conditions include topical or intralesional corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA) for alopecia areata. For tinea capitis, the treatment is topical or oral antifungals. It is important to note

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [80]:
display(Markdown(generate_rag_response(brainTissue_query4)))

Llama.generate: 145 prefix-match hit, remaining 1321 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =    3377.77 ms /  1321 tokens (    2.56 ms per token,   391.09 tokens per second)
llama_perf_context_print:        eval time =    3227.16 ms /   127 runs   (   25.41 ms per token,    39.35 tokens per second)
llama_perf_context_print:       total time =    6658.36 ms /  1448 tokens
llama_perf_context_print:    graphs reused =        122


I. For mild injuries, discharge and observation are recommended.
II. For moderate and severe injuries, optimization of ventilation, oxygenation, and brain perfusion; treatment of complications such as increased intracranial pressure, seizures, and hematomas; and rehabilitation are recommended.
III. Multiple noncranial injuries, which are likely with motor vehicle crashes and falls, often require simultaneous treatment.
IV. At the injury scene, a clear airway is secured and external bleeding is controlled before the patient is moved. Proper immobilization should be maintained with a c

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [81]:
display(Markdown(generate_rag_response(legFracture_query5)))

Llama.generate: 145 prefix-match hit, remaining 1353 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =    3399.06 ms /  1353 tokens (    2.51 ms per token,   398.05 tokens per second)
llama_perf_context_print:        eval time =    3251.64 ms /   127 runs   (   25.60 ms per token,    39.06 tokens per second)
llama_perf_context_print:       total time =    6725.35 ms /  1480 tokens
llama_perf_context_print:    graphs reused =        122


Based on the context, the necessary precautions for a person who has fractured their leg during a hiking trip include immobilizing the injury immediately to prevent further damage to soft tissues and decrease pain. The person should also be treated for hemorrhagic shock if necessary, and any arterial injuries should be repaired surgically unless they affect only small arteries with good collateral circulation. Nerve injuries should be observed, and supportive measures and sometimes physical therapy may be indicated for neuropraxia and axonotmesis.

The treatment steps for a fractured leg include spl

### observations
* the first phase gave generic anaswers from llm
* second phase, we gave some prompts to direct llm to think like the persona given
* in the above phase(3) we gave context from merck manual..which is giving answers from the context.. which is a RAG impelemtation

* the answers seems to be truncated , because of the functions and the parameters we used.

##### lets try to finetune them and see how answers will be generated . the parameters of the llm can be finetunes inorder to extract good answers . it all depends on how good a context we pass to the model


### Fine-tuning

In [ ]:
display(Markdown(generate_rag_response(sepsis_query1,max_tokens=200))) # trying with more tokens
#  tried 300 or 256 etc as there are only 6 points increasing this will not lead to a different answer or long answer

Llama.generate: 1339 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =    3760.10 ms /   143 runs   (   26.29 ms per token,    38.03 tokens per second)
llama_perf_context_print:       total time =    3834.83 ms /   144 tokens
llama_perf_context_print:    graphs reused =        137


Based on the context, the protocol for managing sepsis in a critical care unit includes:
1. Suspecting sepsis or septic shock based on symptoms such as shaking chills, persistent fever, altered sensorium, hypotension, and GI symptoms.
2. Obtaining cultures of blood and any other appropriate specimens.
3. Giving empiric antibiotics after appropriate cultures are obtained.
4. Adjusting antibiotics according to the results of culture and susceptibility testing.
5. Surgically draining any abscesses.
6. Removing any internal devices that are the suspected source of bacteria.

In [89]:
display(Markdown(generate_rag_response(appendicitis_query2, max_tokens=500,temperature=0.3 ))) # trying with more tokens and some temperature

Llama.generate: 1524 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   10387.41 ms /   395 runs   (   26.30 ms per token,    38.03 tokens per second)
llama_perf_context_print:       total time =   10834.43 ms /   396 tokens
llama_perf_context_print:    graphs reused =        382


The common symptoms for appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia, which later shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point. Additional signs include pain felt in the right lower quadrant with palpation of the left lower quadrant (Rovsing sign), an increase in pain from passive extension of the right hip joint that stretches the iliopsoas muscle (psoas sign), or pain caused by passive internal rotation of the flexed thigh (obturator sign). Low-grade fever is common. However, these classic findings appear in less than 50% of patients, and many variations of symptoms and signs occur.

Appendicitis cannot be cured via medicine alone. The standard treatment for appendicitis is surgical removal of the appendix, which can be done through open or laparoscopic appendectomy. If the appendix is perforated, antibiotics should be given to prevent peritonitis. If surgery is impossible, antibiotics can improve the survival rate but are not curative. If a large inflammatory mass is found involving the appendix, terminal ileum, and cecum, resection of the entire mass and ileocolostomy are preferable. In late cases where a pericolic abscess has already formed, the abscess is drained either by an ultrasound-guided percutaneous catheter or by open operation. A Meckel's diverticulum in a patient under the age of 40 should be removed concomitantly with the appendectomy unless extensive inflammation around the appendix prevents the procedure.

##### Observations with increased token count and temperature
* increasing the number of tokens gives big answer in this scenario, but also hits th emax limit
* The good answer came when we used a temperature of 0.3

In [ ]:
display(Markdown(generate_rag_response(hairLoss_query3, max_tokens=500,temperature=0.2 ,k=7))) # trying with more tokens and some temperature
# after multiple tries the context i snot sent full so the model gave below message
# The context does not provide information on the specific treatments or solutions for addressing sudden patchy hair loss due to alopecia areata

#here the no of tokens in the function was 2048 and i got error of max limit reached, so increased to 4096 for better output on m1 mac.

# max_tokens=500,temperature=0.2 ,k=7 these parameters gave a good answer


Llama.generate: 3236 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    6124.29 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   11984.31 ms /   372 runs   (   32.22 ms per token,    31.04 tokens per second)
llama_perf_context_print:       total time =   12358.19 ms /   373 tokens
llama_perf_context_print:    graphs reused =        360


Based on the context, the condition described in the question is alopecia areata. The context mentions that alopecia areata is a sudden patchy hair loss in people with no obvious skin or systemic disorder. It is thought to be an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers.

The context also mentions that treatment for alopecia areata includes topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). It is important to note that the effectiveness of these treatments may vary from person to person.

The possible causes of alopecia areata mentioned in the context include autoimmune disorders and unclear environmental triggers. Other causes of hair loss, such as androgenetic alopecia, drugs, infection, and systemic illnesses, are not mentioned as possible causes of the sudden patchy hair loss described in the question.

Therefore, the answer to the question would be: The effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). The possible causes behind this condition are believed to be autoimmune disorders and unclear environmental triggers.

In [109]:
display(Markdown(generate_rag_response(brainTissue_query4, max_tokens=500,temperature=0.2 ,top_p=0.98, top_k=20)))

# top_p=0.98, top_k=20 these parameters gave a good answer
# without these parameters, the context was not helping much in getting a good answer, adding higher values of K  helped

Llama.generate: 145 prefix-match hit, remaining 1321 prompt tokens to eval
llama_perf_context_print:        load time =    6124.29 ms
llama_perf_context_print: prompt eval time =    3341.76 ms /  1321 tokens (    2.53 ms per token,   395.30 tokens per second)
llama_perf_context_print:        eval time =    9196.04 ms /   359 runs   (   25.62 ms per token,    39.04 tokens per second)
llama_perf_context_print:       total time =   12886.67 ms /  1680 tokens
llama_perf_context_print:    graphs reused =        346


I. For mild injuries, discharge and observation are recommended.
II. For moderate and severe injuries, optimization of ventilation, oxygenation, and brain perfusion; treatment of complications (such as increased intracranial pressure, seizures, and hematomas); and rehabilitation are recommended.
III. Multiple noncranial injuries, which are likely with motor vehicle crashes and falls, often require simultaneous treatment.
IV. At the injury scene, a clear airway is secured and external bleeding is controlled before the patient is moved. Proper immobilization should be maintained with a cervical collar and long spine board until stability of the entire spine has been established. After the initial rapid neurologic assessment, pain should be relieved with a short-acting opioid.
V. In the hospital, after quick initial evaluation, neurologic findings (GCS and pupillary reaction), BP, pulse, and temperature should be recorded frequently for several hours because any deterioration demands prompt attention. Serial GCS and CT results stratify injury severity, which helps guide treatment.
VI. The cornerstone of management for all patients is maintenance of adequate ventilation, oxygenation, and brain perfusion to avoid secondary brain insult.

Therefore, the recommended treatments for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, include optimization of ventilation, oxygenation, and brain perfusion; treatment of complications; rehabilitation; securing a clear airway and controlling external bleeding at the injury scene; and recording neurologic findings and maintaining adequate ventilation, oxygenation, and brain perfusion in the hospital.

In [110]:
display(Markdown(generate_rag_response(legFracture_query5, max_tokens=500,temperature=0.2 ,top_p=0.98, top_k=20)))


Llama.generate: 145 prefix-match hit, remaining 1353 prompt tokens to eval
llama_perf_context_print:        load time =    6124.29 ms
llama_perf_context_print: prompt eval time =    3447.53 ms /  1353 tokens (    2.55 ms per token,   392.45 tokens per second)
llama_perf_context_print:        eval time =    7003.70 ms /   273 runs   (   25.65 ms per token,    38.98 tokens per second)
llama_perf_context_print:       total time =   10700.53 ms /  1626 tokens
llama_perf_context_print:    graphs reused =        263


Based on the context, the necessary precautions for a person who has fractured their leg during a hiking trip include immobilizing the injury immediately to prevent further damage and decrease pain, treating pain typically with opioids, and seeking medical attention for life- or limb-threatening injuries. The treatment steps include splinting or casting for closed reductions, surgical repair for open reductions or severe injuries, and RICE (rest, ice, compression, and elevation) for soft-tissue injuries. For hip surgery rehabilitation, rehabilitation should be started as soon as possible after surgery, with initial goals being to increase strength and prevent atrophy on the unaffected side. Gradual mobilization of the affected limb usually results in full ambulation. Ambulation exercises are started using parallel bars, and as patients progress, they use a walker, crutches, or cane and then walk without devices. Patients should also learn special techniques for climbing stairs and stepping over curbs. Transfer training may be necessary for those who cannot transfer independently from bed to chair, chair to commode, or chair to a standing position. Occupational therapy may focus on self-care activities and improvement of fine motor coordination of muscles and joints, particularly in the upper extremities.

*  while trying multiple iterations on the parameters (details present as code comments) , we learned that the context ,temperature and K values are playing important role for RAG.
* max tokens can give the truncated responses, based on the topic we have to choos a good max token value
* providing a larger context extracted from the vector db is very improtant if the questions tend to be more generic ,rather than specific.
* if specific lower K values are ok, but generic content we need to try higher K values

## Output Evaluation

In [ ]:
groundedness_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score.
"""

In [ ]:
relevance_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects are contained in the answer when evaluating relevance.

Instructions:
1. First write down the steps that are needed to evaluate the context as per the metric.
2. Give a step-by-step explanation if the context adheres to the metric considering the question as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the context using the evaluaton criteria and assign a score.
"""

In [ ]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [ ]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

## Actionable Insights and Business Recommendations

<font size=6 color='blue'>Power Ahead</font>
___